<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 컴퓨터 비전 · 07. YOLOv1

## You Only Look Once: Unified, Real-Time Object Detection

- **원 논문:** [You Only Look Once: Unified, Real-Time Object Detection](https://arxiv.org/abs/1506.02640)
- **저자 / 발표:** Joseph Redmon, Santosh Divvala, Ross Girshick, Ali Farhadi · CVPR 2016 (2015)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch tensor, 합성곱, 역전파의 기초
- **로컬 학습 데이터:** `data/field_curriculum/vision_shapes.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** grid target encoding과 coordinate/object/class 결합 손실을 재현한다.

**축소하거나 생략한 부분:** S=7, B=2, C=20과 24-conv backbone 대신 S=4, B=1, C=4를 사용한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2 Fig. 2: S×S×(B·5+C) | Task 1 encoder | 4×4×9 |
| §2 Eq. (1): class confidence | Task 1 | product assert |
| §2.2 Eq. (3): λcoord=5, λnoobj=.5 | Task 2 | perfect loss 0 |
| §2.3: single evaluation | Task 3 | loss optimization |

## 핵심 아이디어와 수식

$$L=\lambda_{coord}L_{xywh}+L_{obj}+\lambda_{noobj}L_{noobj}+L_{class}$$

**기호 해설:** xywh는 box, obj/noobj는 confidence 항, class는 조건부 class 확률 항이다.

**코드 대응:** Task 1이 grid target, Task 2가 YOLONetwork, Task 3이 네 손실 항을 직접 계산한다.

**입력과 출력 shape:** 입력 [B,1,16,16] → prediction [B,4,4,9], 마지막 9=4 box/conf+4 classes.

**포트폴리오 구현 범위:** B=1과 작은 backbone이라 다중 물체/NMS는 제외하지만 unified loss는 보존한다.

아래 코드는 결과만 내는 라이브러리 호출로 핵심을 감추지 않는다. config, 논문 고유
연산, 모델의 `forward`, 목적함수, `train_step`, 평가 함수를 순서대로 직접 작성한다.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import itertools
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/vision_shapes.npz")
raw = np.load(dataset_path)
images = torch.from_numpy(raw["images"]).float()
labels = torch.from_numpy(raw["labels"]).long()
masks = torch.from_numpy(raw["masks"]).float()
boxes = torch.from_numpy(raw["boxes"]).float()
train_idx = torch.from_numpy(raw["train_idx"][:128]).long()
test_idx = torch.from_numpy(raw["test_idx"][:64]).long()
images, labels, masks, boxes = ACCELERATOR.move(
    images,
    labels,
    masks,
    boxes,
)
train_idx, test_idx = ACCELERATOR.move(train_idx, test_idx)
class_names = raw["class_names"].tolist()
x_train = images[train_idx]
y_train = labels[train_idx]
assert images.shape == (480, 1, 16, 16)
assert masks.shape == (480, 16, 16)
assert boxes.shape == (480, 4)
print(ACCELERATOR.summary())
print(dataset_path.name, images.shape, class_names)

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §2 Fig. 2: S×S×(B·5+C)
- **노트북에서 구현할 부분:** Task 1 encoder
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** 4×4×9

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
@dataclass
class YOLOConfig:
    grid_size: int = 4
    num_classes: int = 4
    coordinate_weight: float = 5.0
    no_object_weight: float = 0.5
    learning_rate: float = 0.015
    steps: int = 18


def encode_yolo_targets(boxes, labels, config):
    batch = len(boxes)
    channels = 5 + config.num_classes
    target = torch.zeros(
        batch,
        config.grid_size,
        config.grid_size,
        channels,
        device=boxes.device,
    )
    centers = (boxes[:, :2] + boxes[:, 2:]) / 2
    sizes = boxes[:, 2:] - boxes[:, :2]
    grid_xy = (centers * config.grid_size).long()
    grid_xy = grid_xy.clamp(0, config.grid_size - 1)
    rows = grid_xy[:, 1]
    columns = grid_xy[:, 0]
    offsets = centers * config.grid_size - grid_xy.float()
    indices = torch.arange(batch, device=boxes.device)
    target[indices, rows, columns, :2] = offsets
    target[indices, rows, columns, 2:4] = sizes.sqrt()
    target[indices, rows, columns, 4] = 1.0
    target[indices, rows, columns, 5 + labels] = 1.0
    return target


config = YOLOConfig()
encoded = encode_yolo_targets(boxes[:2], labels[:2], config)
assert encoded.shape == (2, 4, 4, 9)
assert torch.allclose(encoded[..., 4].sum(dim=(1, 2)), encoded.new_ones(2))

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §2 Eq. (1): class confidence
- **노트북에서 구현할 부분:** Task 1
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** product assert

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
class YOLONetwork(nn.Module):
    def __init__(self, config):
        super().__init__()
        output_channels = 5 + config.num_classes
        self.backbone = nn.Sequential(
            nn.Conv2d(1, 12, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(12, 20, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(20, 20, 3, padding=1),
            nn.ReLU(),
        )
        self.prediction = nn.Conv2d(20, output_channels, kernel_size=1)

    def forward(self, inputs):
        features = self.backbone(inputs)
        prediction = self.prediction(features)
        return prediction.permute(0, 2, 3, 1)


model = YOLONetwork(config).to(DEVICE)
assert model(images[:2]).shape == (2, 4, 4, 9)

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §2.3: single evaluation
- **노트북에서 구현할 부분:** Task 3
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** loss optimization

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

In [ ]:
def yolo_loss(prediction, target, config):
    object_mask = target[..., 4] > 0
    no_object_mask = ~object_mask
    coordinate = F.mse_loss(
        prediction[..., :4][object_mask],
        target[..., :4][object_mask],
    )
    object_loss = F.mse_loss(
        prediction[..., 4][object_mask],
        target[..., 4][object_mask],
    )
    no_object = F.mse_loss(
        prediction[..., 4][no_object_mask],
        target[..., 4][no_object_mask],
    )
    class_loss = F.mse_loss(
        prediction[..., 5:][object_mask],
        target[..., 5:][object_mask],
    )
    return (
        config.coordinate_weight * coordinate
        + object_loss
        + config.no_object_weight * no_object
        + class_loss
    )


def train_yolo_step(model, optimizer, inputs, target, config):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss = yolo_loss(model(inputs), target, config)
    loss.backward()
    optimizer.step()
    return float(loss.detach())


def evaluate_yolo(model, inputs, target):
    model.eval()
    with torch.no_grad():
        prediction = model(inputs)
        predicted_cells = prediction[..., 4].flatten(1).argmax(dim=1)
    expected_cells = target[..., 4].flatten(1).argmax(dim=1)
    return float((predicted_cells == expected_cells).float().mean())


target = encode_yolo_targets(boxes[train_idx], y_train, config)
optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
losses = []
for step in range(config.steps):
    losses.append(train_yolo_step(model, optimizer, x_train, target, config))
test_target = encode_yolo_targets(boxes[test_idx], labels[test_idx], config)
cell_accuracy = evaluate_yolo(model, images[test_idx], test_target)
assert min(losses[3:]) < losses[0]
assert 0.0 <= cell_accuracy <= 1.0
plt.figure(figsize=(5, 3))
plt.plot(losses)
plt.title(f"YOLO cell accuracy={cell_accuracy:.1%}")
plt.tight_layout()
plt.show()

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> S=7, B=2, C=20과 24-conv backbone 대신 S=4, B=1, C=4를 사용한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.